# Diabetic Retinopathy Detection (APTOS 2019)
**Goal:** classify retinal fundus images into 5 severity grades (0 = No DR ... 4 = Proliferative DR) for early detection of diabetic retinopathy.
**Concepts:** CNN (baseline) and Transfer Learning (EfficientNetB0), plus retina-specific preprocessing, Quadratic Weighted Kappa, and Grad-CAM explainability.
**Dataset:** APTOS 2019 Blindness Detection (Kaggle), 3,662 labelled images.

> Educational prototype only. Not a medical diagnostic tool.

**Runtime:** use GPU (Runtime > Change runtime type > T4).

## 1. Setup and data download

In [ ]:
!pip -q install kaggle opencv-python-headless joblib tqdm
import os, zipfile, glob, time, warnings
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             cohen_kappa_score, accuracy_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight
from joblib import Parallel, delayed
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED, IMG, BATCH = 42, 224, 32
np.random.seed(SEED); tf.random.set_seed(SEED)
CLASS_NAMES = ["No DR (0)", "Mild (1)", "Moderate (2)", "Severe (3)", "Proliferative (4)"]
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Download only if the data is not already in this Colab session.
BASE = "/content/aptos"
DATA = f"{BASE}/data"
os.makedirs(BASE, exist_ok=True)

if not os.path.exists(f"{DATA}/train.csv"):
    from getpass import getpass
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ")   # never hard-code the token
    zip_path = f"{BASE}/aptos2019-blindness-detection.zip"
    if not os.path.exists(zip_path):
        !kaggle competitions download -c aptos2019-blindness-detection -p {BASE}
    !unzip -q -n {zip_path} "train_images/*" "train.csv" -d {DATA}    # only what we need
else:
    print("Data already present.")

df = pd.read_csv(f"{DATA}/train.csv")
df["path"] = df["id_code"].apply(lambda x: f"{DATA}/train_images/{x}.png")
print(df.shape)
print(df["diagnosis"].value_counts().sort_index())

## 2. Class distribution
The data is heavily imbalanced (about half the images are "No DR"; Severe has only 193). We handle this with softened class weights later.

In [ ]:
counts = df["diagnosis"].value_counts().sort_index()
plt.figure(figsize=(6,3.5))
plt.bar(CLASS_NAMES, counts.values, color="steelblue")
plt.xticks(rotation=20); plt.ylabel("Images"); plt.title("APTOS 2019 class distribution")
for i, v in enumerate(counts.values): plt.text(i, v+15, str(v), ha="center")
plt.tight_layout(); plt.show()

## 3. Retina-specific preprocessing (done once, cached)
1. **Crop** the black borders around the eye.
2. **Resize** to 224x224.
3. **Ben Graham's method:** subtract a Gaussian-blurred copy to remove lighting differences and make lesions (microaneurysms, exudates, haemorrhages) stand out.

Doing this once and caching to disk makes each training epoch take seconds instead of minutes.

In [ ]:
def crop_black(img, tol=7):
    mask = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) > tol
    if not mask.any(): return img
    rows, cols = np.where(mask.any(1))[0], np.where(mask.any(0))[0]
    return img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]

def preprocess_raw(path, size=IMG):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(crop_black(img), (size, size))
    return img

def ben_graham(img, size=IMG):
    return cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), size/30), -4, 128)

def preprocess(path, size=IMG):
    return ben_graham(preprocess_raw(path, size), size)

CACHE = "/content/aptos_cache.npz"
if os.path.exists(CACHE):
    X = np.load(CACHE)["X"]; print("Loaded cache", X.shape)
else:
    t = time.time()
    X = np.stack(Parallel(n_jobs=-1)(delayed(preprocess)(p) for p in tqdm(df["path"])))
    np.savez_compressed(CACHE, X=X)
    print(f"Preprocessed {X.shape} in {(time.time()-t)/60:.1f} min")
y = df["diagnosis"].values.astype("int32")

In [ ]:
# Before / after preprocessing
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for g in range(5):
    p = df[df.diagnosis == g].sample(1, random_state=1)["path"].iloc[0]
    ax[0, g].imshow(preprocess_raw(p)); ax[0, g].set_title(f"{CLASS_NAMES[g]}\noriginal"); ax[0, g].axis("off")
    ax[1, g].imshow(preprocess(p));     ax[1, g].set_title("preprocessed"); ax[1, g].axis("off")
plt.tight_layout(); plt.show()

## 4. Train/validation split and data pipeline

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_tr.shape, "| Val:", X_val.shape)

# Softened class weights (sqrt of 'balanced'): helps rare classes without wrecking the common ones.
w = compute_class_weight("balanced", classes=np.arange(5), y=y_tr) ** 0.5
CLASS_WEIGHT = {i: float(v) for i, v in enumerate(w)}
print("Class weights:", {k: round(v, 2) for k, v in CLASS_WEIGHT.items()})

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def make_ds(Xa, ya, train):
    ds = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if train: ds = ds.shuffle(2048, seed=SEED)
    ds = ds.batch(BATCH)
    ds = ds.map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = make_ds(X_tr, y_tr, True), make_ds(X_val, y_val, False)

## 5. Helper functions (training curves and evaluation)
We report accuracy, macro-F1 and **Quadratic Weighted Kappa (QWK)**, the official APTOS metric. QWK penalises being far off more (predicting 4 for a true 0) than being one grade off, which suits ordered severity grades.

In [ ]:
def plot_history(histories, title):
    acc = sum([h.history["accuracy"] for h in histories], []); vacc = sum([h.history["val_accuracy"] for h in histories], [])
    loss = sum([h.history["loss"] for h in histories], []);    vloss = sum([h.history["val_loss"] for h in histories], [])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(acc, label="train"); ax[0].plot(vacc, label="val"); ax[0].set_title(f"{title}: accuracy"); ax[0].legend(); ax[0].set_xlabel("epoch")
    ax[1].plot(loss, label="train"); ax[1].plot(vloss, label="val"); ax[1].set_title(f"{title}: loss"); ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()

RESULTS = {}
def evaluate(model, name):
    probs = model.predict(val_ds, verbose=0)
    pred = probs.argmax(1)
    acc = accuracy_score(y_val, pred); f1 = f1_score(y_val, pred, average="macro")
    qwk = cohen_kappa_score(y_val, pred, weights="quadratic")
    RESULTS[name] = {"accuracy": round(acc, 4), "macro_F1": round(f1, 4), "QWK": round(qwk, 4)}
    print(f"{name}: accuracy={acc:.4f} | macro-F1={f1:.4f} | QWK={qwk:.4f}\n")
    print(classification_report(y_val, pred, target_names=CLASS_NAMES, zero_division=0))
    cm = confusion_matrix(y_val, pred)
    ConfusionMatrixDisplay(cm, display_labels=[c.split(" ")[0] for c in CLASS_NAMES]).plot(cmap="Blues", values_format="d")
    plt.title(f"{name}: confusion matrix"); plt.show()
    return probs

## 6. Model A: custom CNN (baseline)

In [ ]:
def build_cnn():
    m = keras.Sequential([
        layers.Input((IMG, IMG, 3)),
        layers.Rescaling(1/255.),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"), layers.Dropout(0.4),
        layers.Dense(5, activation="softmax"),
    ], name="custom_cnn")
    m.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

cnn = build_cnn()
h_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=CLASS_WEIGHT,
                callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
                           keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)])
plot_history([h_cnn], "Custom CNN")
_ = evaluate(cnn, "Custom CNN")

## 7. Model B: transfer learning with EfficientNetB0
- **Stage 1:** freeze the ImageNet-pretrained base, train only the new classification head.
- **Stage 2 (fine-tuning):** unfreeze the top of the base (BatchNorm layers stay frozen) and train with a small learning rate.

In [ ]:
base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dropout(0.3)(x)
out = layers.Dense(5, activation="softmax")(x)
tl_model = keras.Model(base.input, out, name="efficientnetb0_dr")

def compile_tl(lr):
    tl_model.compile(keras.optimizers.Adam(lr), "sparse_categorical_crossentropy", metrics=["accuracy"])

cb = lambda: [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
              keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor="val_loss")]

# Stage 1: head only
compile_tl(1e-3)
h1 = tl_model.fit(train_ds, validation_data=val_ds, epochs=6, class_weight=CLASS_WEIGHT, callbacks=cb())

# Stage 2: fine-tune top layers (BatchNorm stays frozen)
for layer in base.layers[-80:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
compile_tl(3e-5)
h2 = tl_model.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=CLASS_WEIGHT, callbacks=cb())
plot_history([h1, h2], "EfficientNetB0")
probs_tl = evaluate(tl_model, "EfficientNetB0 (transfer learning)")

## 8. Model comparison

In [ ]:
res = pd.DataFrame(RESULTS).T
display(res)
res.plot(kind="bar", figsize=(7, 3.5), rot=0); plt.title("Model comparison (validation set)"); plt.ylim(0, 1); plt.tight_layout(); plt.show()

## 9. Explainability with Grad-CAM
Grad-CAM highlights the regions of the retina that most influenced the prediction. This helps check that the model looks at lesions rather than image artefacts, which is important for trust in a medical setting.

In [ ]:
def gradcam(model, img, class_idx=None):
    gm = keras.Model(model.inputs, [model.get_layer("top_activation").output, model.output])
    xx = tf.cast(img[None], tf.float32)
    with tf.GradientTape() as tape:
        conv, preds = gm(xx)
        if class_idx is None: class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    wts = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv * wts[:, None, None, :], axis=-1))[0]
    cam = (cam / (tf.reduce_max(cam) + 1e-8)).numpy()
    return cam, preds[0].numpy()

def overlay(img, cam, alpha=0.45):
    heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, (img.shape[1], img.shape[0]))), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, heat, alpha, 0)

# one correctly-classified example per grade where possible
pred_val = probs_tl.argmax(1)
fig, ax = plt.subplots(2, 5, figsize=(15, 6.5))
for g in range(5):
    idx = np.where((y_val == g) & (pred_val == g))[0]
    if len(idx) == 0: idx = np.where(y_val == g)[0]
    i = idx[0]
    cam, pr = gradcam(tl_model, X_val[i])
    ax[0, g].imshow(X_val[i]); ax[0, g].set_title(f"True: {CLASS_NAMES[g]}"); ax[0, g].axis("off")
    ax[1, g].imshow(overlay(X_val[i], cam)); ax[1, g].set_title(f"Pred: {CLASS_NAMES[pr.argmax()]} ({pr.max():.0%})"); ax[1, g].axis("off")
plt.suptitle("Grad-CAM: where the model looks", y=1.0); plt.tight_layout(); plt.show()

## 10. Demo: predict on a single fundus image

In [ ]:
def predict_fundus(path):
    img = preprocess(path)
    cam, probs = gradcam(tl_model, img)
    g = int(probs.argmax())
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(preprocess_raw(path)); ax[0].set_title("Input"); ax[0].axis("off")
    ax[1].imshow(overlay(img, cam)); ax[1].set_title(f"Prediction: {CLASS_NAMES[g]}  ({probs[g]:.0%})"); ax[1].axis("off")
    ax[2].barh(CLASS_NAMES, probs, color="steelblue"); ax[2].set_xlim(0, 1); ax[2].set_title("Class probabilities")
    plt.tight_layout(); plt.show()
    print("Educational prototype, not a medical diagnosis.")

# Try a few random images from the dataset (replace with any fundus image path)
for p in df.sample(3, random_state=7)["path"]:
    predict_fundus(p)

## 11. Save the model

In [ ]:
OUT = "/content/aptos_output"; os.makedirs(OUT, exist_ok=True)
tl_model.save(f"{OUT}/dr_efficientnetb0.keras")
cnn.save(f"{OUT}/dr_custom_cnn.keras")
res.to_csv(f"{OUT}/model_comparison.csv")
print("Saved to", OUT)

from google.colab import files
files.download(f"{OUT}/dr_efficientnetb0.keras")

## 12. Summary and limitations
- **Preprocessing** (border crop + Ben Graham) and **transfer learning** (EfficientNetB0) are the main drivers of improvement over the baseline CNN.
- The data is **imbalanced** and the middle grades (Moderate and Severe) are hardest to separate; neighbouring-grade confusion is expected and is why QWK is reported.
- Only one 80/20 split was used, and the dataset is from a single source. Real-world use would need external validation, calibration and clinical review.
- **Social relevance:** diabetic retinopathy is a leading cause of preventable blindness; automated screening can help triage patients where ophthalmologists are scarce.
- This is an **educational prototype and not a medical diagnostic tool**.